# Mini-Project 3: Lake Victoria Fish Stock & Export Risk Model

## Purpose
This project explores how harvesting affects fish stocks and how changing
fish prices affect revenue for a fish-export cooperative in Jinja.

## Model assumptions
The initial stock is 4,000 tonnes, the carrying capacity is 10,000 tonnes,
and the growth parameter is 0.4 per weekly step. These are illustrative
assignment values, not measured estimates for Lake Victoria.

The simulation covers 52 weeks. Each week's harvest is calculated from
the stock at the start of that week. Revenue converts harvested tonnes
into kilograms before multiplying by the price in UGX/kg.

## Approach
We first examine why Fibonacci growth is unsuitable for fish stocks.
We then model logistic growth with harvesting, simulate prices, and
compare weekly harvest fractions of 0.05, 0.10, 0.20 and 0.30.

In [1]:
import sys
from pathlib import Path
import numpy as np

# Locate the project folder from either the root or notebooks folder.
project_folder = Path.cwd()
if not (project_folder / "src").is_dir():
    project_folder = project_folder.parent

# Allow Python to find our reusable classes.
sys.path.insert(0, str(project_folder))

from src.fishstock import FishStock

# Create a fish stock object using the assignment's default parameters.
jinja_fish_stock = FishStock("Jinja Fish Cooperative")

# Check the values stored in the object.
print("Cooperative:", jinja_fish_stock.cooperative_name)
print("Initial stock:", jinja_fish_stock.initial_stock, "tonnes")
print("Carrying capacity:", jinja_fish_stock.carrying_capacity, "tonnes")
print("Weekly growth parameter:", jinja_fish_stock.growth_rate)

Cooperative: Jinja Fish Cooperative
Initial stock: 4000.0 tonnes
Carrying capacity: 10000.0 tonnes
Weekly growth parameter: 0.4


## 1. Examine the Fibonacci baseline

We begin with 15 Fibonacci numbers, starting with 0 and 1, as a
hypothetical stock sequence. Each subsequent value is the sum of
the previous two. This baseline has no calibrated biological units.

In [2]:
# Start the Fibonacci sequence with its first two values.
fibonacci_stock = [0, 1]

# Generate the remaining values until the sequence contains 15 numbers.
while len(fibonacci_stock) < 15:
    next_stock = fibonacci_stock[-1] + fibonacci_stock[-2]
    fibonacci_stock.append(next_stock)

# Display the hypothetical stock sequence.
print("Fibonacci stock baseline:", fibonacci_stock)
print("Number of values:", len(fibonacci_stock))

Fibonacci stock baseline: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]
Number of values: 15


### Limitations of Fibonacci stock growth

Fibonacci growth continues without an upper limit, whereas a lake has
limited food, space and other resources. The sequence does not model
reproduction, natural mortality or harvesting explicitly. Its growth
pattern is fixed by a numerical rule rather than estimated from fish
population data. A logistic model with harvesting is more suitable for
this exercise because it includes a carrying capacity and removes the
fish harvested each week.

## 2. Simulate stock growth and harvesting

The weekly model is:

N(t+1) = N(t) + r × N(t) × (1 − N(t)/K) − h × N(t)

Here, N(t) is stock in tonnes, r is the growth parameter, K is carrying
capacity, and h is the fraction harvested each week. Growth and harvest
are both calculated from the stock at the start of the week.

We first simulate 52 weeks using h = 0.10.

In [3]:
import importlib
import src.fishstock as fishstock

# Reload the module and create an object with the new simulation method.
importlib.reload(fishstock)
jinja_fish_stock = fishstock.FishStock("Jinja Fish Cooperative")

# Simulate one year with a weekly harvest of 10% of the starting stock.
stock_levels, weekly_harvest = jinja_fish_stock.simulate(harvest_rate=0.10)

# Check the first week's calculation and the final stock.
print("Stock observations:", len(stock_levels))
print("Weekly harvests:", len(weekly_harvest))
print(f"First week's harvest: {weekly_harvest[0]:,.2f} tonnes")
print(f"Stock after week 1: {stock_levels[1]:,.2f} tonnes")
print(f"Stock after week 52: {stock_levels[-1]:,.2f} tonnes")

Stock observations: 53
Weekly harvests: 52
First week's harvest: 400.00 tonnes
Stock after week 1: 4,560.00 tonnes
Stock after week 52: 7,500.00 tonnes
